In [1]:
import os
import glob
import pickle
import numpy as np
import pandas as pd
from scipy.interpolate import interp1d
from scipy.signal import savgol_filter
from sklearn.preprocessing import MinMaxScaler
import warnings
warnings.filterwarnings('ignore')

# ── Paths ──────────────────────────────────────────
LAB_DIR      = "../data/lab"
EV_DIR       = "../data/ev_real_world"
RESULTS_DIR  = "../results/metrics"
os.makedirs(RESULTS_DIR, exist_ok=True)

# ── Constants ──────────────────────────────────────
EOL_SOH      = 0.80        # 80% capacity = End of Life
WINDOW_SIZE  = 20          # LSTM input: 20 cycles lookback
FEATURES     = ['v_raw_V', 'i_raw_A', 't_cell_degC', 'soc_est']

print("✅ Config loaded")
print(f"   Lab dir:  {LAB_DIR}")
print(f"   EV dir:   {EV_DIR}")
print(f"   EoL:      {EOL_SOH*100}% SoH")
print(f"   Window:   {WINDOW_SIZE} cycles")


✅ Config loaded
   Lab dir:  ../data/lab
   EV dir:   ../data/ev_real_world
   EoL:      80.0% SoH
   Window:   20 cycles


In [2]:
def load_cell(filepath, resolution='30s'):
    """Load a KIT NMC cell CSV and return cleaned DataFrame."""
    df = pd.read_csv(filepath, sep=';')
    df['time_days'] = df['timestamp_s'] / 86400
    df['cell_id']   = os.path.basename(filepath).replace('.csv','')
    df['resolution']= resolution
    return df

def extract_rpt_labels(df):
    """Extract RPT checkpoints: capacity, R0, R1 per EFC."""
    rpt = df[df['cap_aged_est_Ah'].notna()][
        ['EFC','cap_aged_est_Ah','R0_mOhm','R1_mOhm','time_days']
    ].copy().reset_index(drop=True)
    if len(rpt) < 2:
        return None
    rpt['SoH_pct'] = (rpt['cap_aged_est_Ah'] / rpt['cap_aged_est_Ah'].iloc[0]) * 100
    return rpt

def compute_eol_and_rul(rpt):
    """Interpolate EoL EFC at 80% SoH, compute RUL per checkpoint."""
    if rpt is None or len(rpt) < 2:
        return None, None
    # Interpolate/extrapolate EoL
    try:
        if (rpt['SoH_pct'] <= 80).any():
            f = interp1d(rpt['SoH_pct'].values[::-1],
                         rpt['EFC'].values[::-1],
                         kind='linear', fill_value='extrapolate')
            eol_efc = float(f(80.0))
        else:
            coeffs  = np.polyfit(rpt['EFC'], rpt['SoH_pct'], 2)
            efc_ext = np.linspace(rpt['EFC'].max(), rpt['EFC'].max()*4, 2000)
            soh_ext = np.polyval(coeffs, efc_ext)
            hits    = efc_ext[soh_ext <= 80]
            eol_efc = float(hits[0]) if len(hits) > 0 else rpt['EFC'].max() * 2
    except:
        eol_efc = rpt['EFC'].max() * 1.5

    rpt['EoL_EFC'] = eol_efc
    rpt['RUL_efc'] = (eol_efc - rpt['EFC']).clip(lower=0)
    return rpt, eol_efc

# ── Test on one file ────────────────────────────────
test_file = glob.glob(f"{LAB_DIR}/cell_log_age_30s_P067_1*.csv")[0]
df_test   = load_cell(test_file)
rpt_test  = extract_rpt_labels(df_test)
rpt_test, eol = compute_eol_and_rul(rpt_test)

print(f"✅ Loaded: {os.path.basename(test_file)}")
print(f"   Rows: {len(df_test):,}  |  EFCs: {df_test['EFC'].max():.1f}")
print(f"   RPT points: {len(rpt_test)}  |  EoL @ EFC: {eol:.1f}")
print(f"\nRPT Labels:")
print(rpt_test[['EFC','cap_aged_est_Ah','SoH_pct','RUL_efc']].round(2).to_string(index=False))


✅ Loaded: cell_log_age_30s_P067_1_S01_C02.csv
   Rows: 210,404  |  EFCs: 209.2
   RPT points: 5  |  EoL @ EFC: 138.9

RPT Labels:
   EFC  cap_aged_est_Ah  SoH_pct  RUL_efc
  1.11             2.95   100.00   137.78
 19.58             2.86    97.06   119.31
 79.56             2.68    90.76    59.33
147.12             2.31    78.51     0.00
206.50             1.73    58.75     0.00


In [4]:
def process_all_cells(lab_dir, resolution='30s'):
    """Process all lab CSV files and return summary + RPT labels."""
    pattern = f"{lab_dir}/cell_log_age_{resolution}_*.csv"
    files   = sorted(glob.glob(pattern))
    print(f"Found {len(files)} {resolution} files...")

    all_rpt    = []
    skipped    = []

    for fpath in files:
        try:
            df  = load_cell(fpath, resolution)
            rpt = extract_rpt_labels(df)
            rpt, eol = compute_eol_and_rul(rpt)
            if rpt is None:
                skipped.append(os.path.basename(fpath))
                continue
            rpt['cell_id'] = os.path.basename(fpath).replace('.csv','')
            rpt['file']    = fpath
            all_rpt.append(rpt)
        except Exception as e:
            skipped.append(f"{os.path.basename(fpath)}: {e}")

    df_all = pd.concat(all_rpt, ignore_index=True)
    print(f"✅ Processed: {len(all_rpt)} cells")
    print(f"⚠️  Skipped:   {len(skipped)} cells")
    if skipped:
        for s in skipped[:5]:
            print(f"   - {s}")
    return df_all

df_labels = process_all_cells(LAB_DIR, resolution='30s')

# Summary stats
print(f"\n📊 Dataset Summary:")
print(f"   Total labeled RPT rows: {len(df_labels):,}")
print(f"   Unique cells:           {df_labels['cell_id'].nunique()}")
print(f"   RUL range (EFCs):       {df_labels['RUL_efc'].min():.1f} → {df_labels['RUL_efc'].max():.1f}")
print(f"   SoH range (%):          {df_labels['SoH_pct'].min():.1f} → {df_labels['SoH_pct'].max():.1f}")
print(f"   EoL EFC range:          {df_labels['EoL_EFC'].min():.1f} → {df_labels['EoL_EFC'].max():.1f}")

df_labels.to_csv(f"{RESULTS_DIR}/all_cells_rpt_labels.csv", index=False)
print(f"\n💾 Saved → results/metrics/all_cells_rpt_labels.csv")


Found 219 30s files...
✅ Processed: 219 cells
⚠️  Skipped:   0 cells

📊 Dataset Summary:
   Total labeled RPT rows: 3,719
   Unique cells:           219
   RUL range (EFCs):       0.0 → 5925.9
   SoH range (%):          33.2 → 100.0
   EoL EFC range:          38.7 → 5927.1

💾 Saved → results/metrics/all_cells_rpt_labels.csv


In [5]:
def peek_ev_dataset(ev_dir):
    """Inspect EV pkl files and label CSVs."""
    for ds in ['battery_dataset1','battery_dataset2','battery_dataset3']:
        data_path  = f"{ev_dir}/{ds}/data"
        label_path = f"{ev_dir}/{ds}/label/label.csv"

        pkl_files  = glob.glob(f"{data_path}/*.pkl")
        label_df   = pd.read_csv(label_path)

        # Load one sample pkl
        sample     = pickle.load(open(pkl_files[0], 'rb'))

        print(f"\n📦 {ds}")
        print(f"   PKL files:    {len(pkl_files):,}")
        print(f"   Label shape:  {label_df.shape}")
        print(f"   Label cols:   {label_df.columns.tolist()}")
        print(f"   Sample pkl type: {type(sample)}")
        if isinstance(sample, dict):
            print(f"   Sample keys:  {list(sample.keys())}")
        elif isinstance(sample, pd.DataFrame):
            print(f"   Sample cols:  {sample.columns.tolist()}")
            print(f"   Sample shape: {sample.shape}")

peek_ev_dataset(EV_DIR)



📦 battery_dataset1
   PKL files:    459,061
   Label shape:  (217, 3)
   Label cols:   ['Unnamed: 0', 'car', 'label']
   Sample pkl type: <class 'int'>

📦 battery_dataset2
   PKL files:    467,582
   Label shape:  (198, 3)
   Label cols:   ['Unnamed: 0', 'car', 'label']
   Sample pkl type: <class 'int'>

📦 battery_dataset3
   PKL files:    176,327
   Label shape:  (50, 3)
   Label cols:   ['Unnamed: 0', 'car', 'label']
   Sample pkl type: <class 'int'>


In [6]:
import pickle, glob

ds_path = "../data/ev_real_world/battery_dataset1"
pkl_files = sorted(glob.glob(f"{ds_path}/data/*.pkl"))
label_df  = pd.read_csv(f"{ds_path}/label/label.csv")

# Check label structure
print("Label CSV:\n", label_df.head(10))
print("\nLabel stats:\n", label_df['label'].describe())

# Load a few pkls to understand structure
for f in pkl_files[:3]:
    with open(f, 'rb') as fp:
        data = pickle.load(fp)
    print(f"\nFile: {os.path.basename(f)}")
    print(f"  Type: {type(data)}")
    print(f"  Value: {data}")


Label CSV:
    Unnamed: 0  car  label
0           0    0      0
1           1    1      0
2           2    2      0
3           3    3      0
4           4    4      0
5           5    5      0
6           6    6      0
7           7    7      0
8           8    8      0
9           9    9      0

Label stats:
 count    217.000000
mean       0.142857
std        0.350736
min        0.000000
25%        0.000000
50%        0.000000
75%        0.000000
max        1.000000
Name: label, dtype: float64

File: 0.pkl
  Type: <class 'int'>
  Value: 119547037146038801333356

File: 10.pkl
  Type: <class 'int'>
  Value: 119547037146038801333356

File: 100.pkl
  Type: <class 'int'>
  Value: 119547037146038801333356


In [7]:
import pickle, glob, os
import pandas as pd

ds_path = "../data/ev_real_world/battery_dataset1"
pkl_files = sorted(glob.glob(f"{ds_path}/data/*.pkl"))
label_df  = pd.read_csv(f"{ds_path}/label/label.csv")

# 1. Check labelled cars vs pkl file names
print("Total pkl files:", len(pkl_files))
print("Total labelled cars:", len(label_df))

# Sample file names — are they car IDs?
print("\nFirst 10 pkl filenames:")
for f in pkl_files[:10]:
    print(f"  {os.path.basename(f)}")

# 2. Check if pkl names match car column
pkl_ids = [int(os.path.splitext(os.path.basename(f))[0]) for f in pkl_files[:20]]
print("\nPKL IDs (first 20):", pkl_ids)
print("Car IDs in label:", label_df['car'].tolist()[:20])

# 3. Try loading a PKL that matches a label car ID
car_id = label_df['car'].iloc[0]  # car 0
matching_pkl = f"{ds_path}/data/{car_id}.pkl"
if os.path.exists(matching_pkl):
    with open(matching_pkl, 'rb') as f:
        data = pickle.load(f)
    print(f"\nCar {car_id} pkl → type: {type(data)}, value: {data}")
else:
    print(f"\nNo direct match for car {car_id}")

# 4. Try reading a large-numbered pkl (100001.pkl etc)
big_pkl = pkl_files[-1]
with open(big_pkl, 'rb') as f:
    data = pickle.load(f)
print(f"\nLargest pkl: {os.path.basename(big_pkl)}")
print(f"  Type: {type(data)}, Value preview: {str(data)[:200]}")

# 5. Check if there's any other file format in the data folder
all_files = os.listdir(f"{ds_path}/data")
extensions = set([os.path.splitext(f)[1] for f in all_files])
print(f"\nFile extensions in data/: {extensions}")
print(f"Total files: {len(all_files)}")


Total pkl files: 459061
Total labelled cars: 217

First 10 pkl filenames:
  0.pkl
  10.pkl
  100.pkl
  100001.pkl
  100002.pkl
  100003.pkl
  100005.pkl
  100006.pkl
  100007.pkl
  10001.pkl

PKL IDs (first 20): [0, 10, 100, 100001, 100002, 100003, 100005, 100006, 100007, 10001, 100010, 100012, 100013, 100014, 100015, 100016, 100017, 100018, 100019, 100022]
Car IDs in label: [0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19]

Car 0 pkl → type: <class 'int'>, value: 119547037146038801333356

Largest pkl: 99999.pkl
  Type: <class 'int'>, Value preview: 119547037146038801333356

File extensions in data/: {'.pkl'}
Total files: 459061


In [8]:
import torch
import glob, os
import pandas as pd
import numpy as np

def load_ev_snippet(pkl_path):
    """Correct loader for EVBattery pkl files using torch.load()"""
    data = torch.load(pkl_path, map_location='cpu', weights_only=False)
    return data

# Test on one file
ds_path = "../data/ev_real_world/battery_dataset1"
pkl_files = sorted(glob.glob(f"{ds_path}/data/*.pkl"))

# Load and inspect first few
for f in pkl_files[:3]:
    data = load_ev_snippet(f)
    print(f"File: {os.path.basename(f)}")
    print(f"  Type:  {type(data)}")
    if isinstance(data, torch.Tensor):
        print(f"  Shape: {data.shape}  dtype: {data.dtype}")
        print(f"  Sample values: {data[:2]}")
    elif isinstance(data, dict):
        print(f"  Keys: {list(data.keys())}")
    elif isinstance(data, (list, tuple)):
        print(f"  Length: {len(data)}, element type: {type(data[0])}")
    print()


File: 0.pkl
  Type:  <class 'tuple'>
  Length: 2, element type: <class 'numpy.ndarray'>

File: 10.pkl
  Type:  <class 'tuple'>
  Length: 2, element type: <class 'numpy.ndarray'>

File: 100.pkl
  Type:  <class 'tuple'>
  Length: 2, element type: <class 'numpy.ndarray'>



In [10]:
import torch, glob, os, pickle
import numpy as np
from collections import OrderedDict

ds_path = "../data/ev_real_world/battery_dataset1"
pkl_files = sorted(glob.glob(f"{ds_path}/data/*.pkl"))

# Inspect first 3 files
for f in pkl_files[:3]:
    data = torch.load(f, map_location='cpu', weights_only=False)
    arr0, meta = data[0], data[1]

    print(f"── {os.path.basename(f)} ──")
    print(f"  Snippet shape:  {arr0.shape}  dtype={arr0.dtype}")
    print(f"  Meta keys:      {list(meta.keys())}")
    print(f"  Meta values:")
    for k, v in meta.items():
        print(f"    {k}: {v}  (type={type(v).__name__})")
    print(f"  First row of snippet: {arr0[0].round(4)}")
    print()


── 0.pkl ──
  Snippet shape:  (128, 8)  dtype=float64
  Meta keys:      ['label', 'car', 'charge_segment', 'mileage', 'capacity']
  Meta values:
    label: 00  (type=str)
    car: 0  (type=int)
    charge_segment: 1  (type=str)
    mileage: 87968.496  (type=float)
    capacity: 0  (type=int)
  First row of snippet: [  3.7614 -42.7     22.8      3.809    3.625   11.       4.       0.    ]

── 10.pkl ──
  Snippet shape:  (128, 8)  dtype=float64
  Meta keys:      ['label', 'car', 'charge_segment', 'mileage', 'capacity']
  Meta values:
    label: 00  (type=str)
    car: 0  (type=int)
    charge_segment: 5  (type=str)
    mileage: 88488.9984  (type=float)
    capacity: 0  (type=int)
  First row of snippet: [  4.0458 -75.2667  69.8667   4.0927   3.7427  20.3333  17.       0.    ]

── 100.pkl ──
  Snippet shape:  (128, 8)  dtype=float64
  Meta keys:      ['label', 'car', 'charge_segment', 'mileage', 'capacity']
  Meta values:
    label: 00  (type=str)
    car: 0  (type=int)
    charge_segment